In [ ]:
# bgr8转jpeg格式 bgr8 to jpeg format
import enum
import cv2

def bgr8_to_jpeg(value, quality=75):
    """
    将BGR8格式的OpenCV图像转换为JPEG字节流。
    用于在ipywidgets.Image控件中实时显示视频帧。
    
    参数:
        value: OpenCV BGR格式的numpy数组图像
        quality: JPEG压缩质量 (0-100), 默认75
    
    返回:
        bytes: JPEG编码的字节数据
    """
    return bytes(cv2.imencode('.jpg', value)[1])

## 导入相关包，创建相机实例 Import related packages and create a camera instance

In [2]:
import sys
sys.path.append('/home/pi/project_demo/lib')
#导入麦克纳姆小车驱动库 Import Mecanum Car Driver Library
from McLumk_Wheel_Sports import *

import cv2
import mediapipe as mp
import ipywidgets.widgets as widgets
import threading
import time
import sys
import math

image_widget = widgets.Image(format='jpeg', width=640, height=480)

## 创建相关控制变量 Create relevant control variables

In [3]:
global face_x, face_y, face_w, face_h
face_x = face_y = face_w = face_h = 0
global target_valuex
target_valuex = 2048
global target_valuey
target_valuey = 2048

## 创建PID控制实例 Create a PID control instance

In [4]:
import PID
#xservo_pid = PID.PositionalPID(1.1, 0.4, 0.01)#1.1 0.4 0.01
direction_pid = PID.PositionalPID(0.8, 0, 0.2)
yservo_pid = PID.PositionalPID(0.8, 0.2, 0.01)
speed_pid = PID.PositionalPID(1.1, 0, 0.2)


In [ ]:
# 定义 target_servox 和 target_servoy 在外部 Define target_servox and target_servoy externally
target_servox = 90
target_servoy = 25

def servo_reset():
    """
    复位舵机到初始位置。
    舵机1复位到90°（水平居中），舵机2复位到80°（仰角初始位置）。
    在程序启动和结束时调用，确保云台回到默认姿态。
    """
    bot.Ctrl_Servo(1, 90)
    bot.Ctrl_Servo(2, 80)

servo_reset()

## 创建结束进程函数 Create a process termination function

In [ ]:
# 线程功能操作库 Thread function operation library
import inspect
import ctypes

def _async_raise(tid, exctype):
    """
    向指定线程异步注入异常以强制终止线程。
    通过调用Python C API的PyThreadState_SetAsyncExc实现。
    
    参数:
        tid: 目标线程的标识符 (thread.ident)
        exctype: 要注入的异常类型 (如 SystemExit)
    
    异常:
        ValueError: 当线程ID无效时抛出
    """
    tid = ctypes.c_long(tid)
    if not inspect.isclass(exctype):
        exctype = type(exctype)
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(exctype))
    if res == 0:
        raise ValueError("invalid thread id")
    elif res != 1:
        # 如果返回值大于1，需要用NULL参数再次调用以恢复状态
        ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)

def stop_thread(thread):
    """
    终止指定的线程。通过向线程注入SystemExit异常实现。
    用于安全结束Face_Follow后台线程，释放摄像头资源。
    
    参数:
        thread: 要终止的threading.Thread对象
    """
    _async_raise(thread.ident, SystemExit)

## 人体姿态识别函数 Body pose recognition function

In [ ]:
class BodyDetector:
    """
    人体姿态检测器类。封装MediaPipe Pose模型和绘图工具。

    使用方法:
        detector = BodyDetector(0.75)
        results = detector.pose.process(img_RGB)
        frame = detector.fancyDraw(frame, bbox)
    """

    def __init__(self, minDetectionCon=0.5):
        """
        初始化人体姿态检测器。

        参数:
            minDetectionCon: 最小检测置信度 (0.0~1.0)，低于此值的结果将被忽略。
                             默认0.5，调高可减少误检但可能漏检。
        """
        self.mpPose = mp.solutions.pose
        self.mpDraw = mp.solutions.drawing_utils
        self.pose = self.mpPose.Pose(
            min_detection_confidence=minDetectionCon,
            min_tracking_confidence=0.5
        )

    def draw_pose(self, frame, pose_landmarks):
        """
        在画面上绘制完整的33个姿态关键点和骨架连接线。
        """
        self.mpDraw.draw_landmarks(
            frame, pose_landmarks, self.mpPose.POSE_CONNECTIONS)

    def fancyDraw(self, frame, bbox, l=30, t=5):
        """
        用角线风格绘制包围盒（四角L形标记），比全框更美观。

        参数:
            frame: 要绘制到的图像
            bbox:  包围盒 (x, y, w, h)
            l:     角线长度（像素），默认30
            t:     角线粗细（像素），默认5

        返回:
            frame: 绘制后的图像
        """
        x, y, w, h = bbox
        x1, y1 = x + w, y + h
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        # 左上角
        cv2.line(frame, (x, y), (x + l, y), (0, 255, 0), t)
        cv2.line(frame, (x, y), (x, y + l), (0, 255, 0), t)
        # 右上角
        cv2.line(frame, (x1, y), (x1 - l, y), (0, 255, 0), t)
        cv2.line(frame, (x1, y), (x1, y + l), (0, 255, 0), t)
        # 左下角
        cv2.line(frame, (x, y1), (x + l, y1), (0, 255, 0), t)
        cv2.line(frame, (x, y1), (x, y1 - l), (0, 255, 0), t)
        # 右下角
        cv2.line(frame, (x1, y1), (x1 - l, y1), (0, 255, 0), t)
        cv2.line(frame, (x1, y1), (x1, y1 - l), (0, 255, 0), t)
        return frame

#  开启摄像头 Turn on the camera

In [8]:
image = cv2.VideoCapture(0)
image_width = 640
image_height = 480
image.set(3, image_width)
image.set(4, image_height)
image_width = image.get(cv2.CAP_PROP_FRAME_WIDTH)
image_height = image.get(cv2.CAP_PROP_FRAME_HEIGHT)
# image.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter.fourcc('M', 'J', 'P', 'G'))
# image.set(cv2.CAP_PROP_BRIGHTNESS, 62) #设置亮度 -64 - 64  0.0 Set Brightness -64 - 64 0.0
# image.set(cv2.CAP_PROP_CONTRAST, 63) #设置对比度 -64 - 64  2.0 Set Contrast -64 - 64 2.0
# image.set(cv2.CAP_PROP_EXPOSURE, 4800) #设置曝光值 1.0 - 5000  156.0 Set the exposure value 1.0 - 5000 156.0

#csi
# from picamera2 import Picamera2, Preview
# import libcamera
# picam2 = Picamera2()  
# camera_config = picam2.create_preview_configuration(main={"format":'RGB888',"size":(320,240)})
# camera_config["transform"] = libcamera.Transform(hflip=1, vflip=1)
# picam2.configure(camera_config) 
# picam2.start()  

带死区控制，通过人体包围盒高度控制距离，水平/垂直偏差控制舵机转动 With dead zone control, distance is controlled by body bounding box height, horizontal/vertical deviation controls servo rotation

In [ ]:
# ==================== 线程间共享数据 Shared data between threads ====================
shared_lock = threading.Lock()
shared_frame = None           # 最新读取的原始画面帧（BGR格式）
shared_landmarks = None       # 最新检测到的姿态关键点集（NormalizedLandmarkList或None）
shared_ready = False          # True表示有新的检测结果待消费

# 躯干关键点索引：双肩+双胯，用于计算稳定的人体中心
TORSO = [11, 12, 23, 24]


def Pose_Detect():
    """
    姿态检测线程。独立死循环，专责从摄像头读取画面并运行MediaPipe Pose检测，
    将原始帧和关键点结果写入共享缓冲区。
    """
    global shared_frame, shared_landmarks, shared_ready
    body_detector = BodyDetector(0.75)
    while 1:
        ret, frame = image.read()
        if not ret:
            continue
        img_RGB = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = body_detector.pose.process(img_RGB)
        with shared_lock:
            shared_frame = frame
            shared_landmarks = results.pose_landmarks
            shared_ready = True


def Body_Follow():
    """
    人体跟踪主循环。从共享缓冲区读取姿态关键点，直接计算躯干包围盒后执行PID控制。

    一条龙完成：取关键点 → 算包围盒/中心 → 画骨架 → PID → 控制小车 → 显示画面
    """
    global x, w, y, h, shared_ready
    speed = 30
    body_detector = BodyDetector(0.75)
    while 1:
        with shared_lock:
            if not shared_ready or shared_frame is None:
                continue
            frame = shared_frame.copy()
            landmarks = shared_landmarks
            shared_ready = False

        # ── 计算躯干包围盒与中心（原 findBody 逻辑）─────────────────
        bbox = 0, 0, 0, 0
        center_x = center_y = 0
        person_detected = False

        if landmarks:
            person_detected = True
            ih, iw, _ = frame.shape

            # 仅取躯干4点：11左肩, 12右肩, 23左胯, 24右胯
            xs, ys = [], []
            for idx in TORSO:
                lm = landmarks.landmark[idx]
                xs.append(int(lm.x * iw))
                ys.append(int(lm.y * ih))

            x_min, x_max = min(xs), max(xs)
            y_min, y_max = min(ys), max(ys)
            bbox = x_min, y_min, x_max - x_min, y_max - y_min
            center_x = sum(xs) // len(xs)
            center_y = sum(ys) // len(ys)

            # 绘制完整骨架 + 躯干包围盒
            body_detector.draw_pose(frame, landmarks)
            frame = body_detector.fancyDraw(frame, bbox)

        x, y, w, h = bbox
        # ── 包围盒计算结束 ─────────────────────────────────────────

        if person_detected:
            # 水平PID：躯干中心对准画面中心
            direction_pid.SystemOutput = center_x
            direction_pid.SetStepSignal(int(image_width / 2))
            direction_pid.SetInertiaTime(0.01, 0.05)
            target_valuex = int(direction_pid.SystemOutput + 65)

            # 垂直PID：躯干中心对准画面中心（死区±40px）
            if math.fabs(int(image_height / 2) - (y + h / 2)) > 40:
                yservo_pid.SystemOutput = y + h / 2
                yservo_pid.SetStepSignal(int(image_height / 2))
                yservo_pid.SetInertiaTime(0.01, 0.05)
                target_valuey = int(850 + yservo_pid.SystemOutput)
                target_servoy = int((target_valuey - 500) / 10)
                target_servoy = max(0, min(100, target_servoy))
                bot.Ctrl_Servo(2, target_servoy)

            # 速度PID：躯干高度 → 距离（目标 = 画面高度的20%，即 480*0.2 = 96px）
            speed_pid.SystemOutput = int(h)
            speed_pid.SetStepSignal(96)
            speed_pid.SetInertiaTime(0.01, 0.1)
            speed_value = min(255, max(0, int(speed_pid.SystemOutput)))

            # 画面叠加信息
            cv2.putText(frame, f"torso_h {int(h)} target_x {target_valuex}",
                        (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

            # 小车运动决策（躯干高度目标96px，死区 ±10px）
            if target_valuex > 50:
                rotate_left(int(speed / 5))
            elif target_valuex < -50:
                rotate_right(int(speed / 5))
            elif 86 < h < 106:
                stop_robot()
            elif h > 106 and abs(target_valuex) < 30:
                move_backward(speed)
            elif h < 86 and abs(target_valuex) < 30:
                move_forward(speed_value)
            else:
                stop_robot()
        else:
            stop_robot()

        try:
            image_widget.value = bgr8_to_jpeg(frame)
        except:
            continue

In [ ]:
display(image_widget)

# 启动姿态检测线程（专责MediaPipe Pose检测）
thread_pose = threading.Thread(target=Pose_Detect)
thread_pose.daemon = True
thread_pose.start()

# 启动跟踪控制线程（PID计算与小车控制）
thread_ctrl = threading.Thread(target=Body_Follow)
thread_ctrl.daemon = True
thread_ctrl.start()

# picam2.stop()
# picam2.close()

In [ ]:
# 结束进程，释放摄像头，需要结束时执行
# End the process, release the camera, and execute when it is finished
stop_thread(thread_ctrl)
stop_thread(thread_pose)
# 释放摄像头资源 Release camera resources
image.release()
# 复位舵机 Reset servo
bot.Ctrl_Servo(1, 90)
bot.Ctrl_Servo(2, 25)